<a href="https://colab.research.google.com/github/saad0O5/FlyRank-Internship-Work/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad0O5/FlyRank-s-Project/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring**

I explored all four predefined lanes plus the freestyle AI-referral direction on the starter
data before committing. Lane 1 (signal analysis) showed almost no correlation between CTR and
search_volume, impressions, or position. Lane 3 (clustering) hit a missing-data artifact —
engagement_rate collapsed to exactly 0 for every content_type, which the data dictionary warns
is a sign of missingness, not a real signal. Lane 4 (CTR/engagement scoring) had a genuine CTR
drop by position tier, but the "under-performer" list was dominated by a tied block of pages
with ctr = 0, a floor effect that would need extra design work before it could rank anything
meaningfully.

**Lane 2** held up best under scrutiny. A decision tree trained on all available features
independently rediscovered my original finding: declining pages skew younger (the tree's key
split lands at content_age_days <= 312.5 days). Content_type also shows a real, separate effect —
keyword and comparison articles decline at roughly double the rate of feedly articles, and this
gap holds across age tiers, so it isn't just age in disguise. I also found decline is unevenly
concentrated across clients (one client at 93.7% decline, another at 0%), which I want to
account for as I build toward a ranked queue.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/saad0O5/FlyRank-Internship-Work"
REPO_DIR = "FlyRank-Internship-Work"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print(df.shape)

(30000, 45)


## 2. The question: decision, action, cost of a wrong call

**Task type: Ranking / scoring** — "which pages should be reviewed first?", not a plain yes/no
classification. Success metric: Precision@K (K = reviewer capacity, e.g. top 20 or 50).

**Decision this improves:** which pages a content reviewer should look at first, out of many
candidates, when review time is limited.

**Who acts on it:** a content/SEO reviewer who can only manually review a small number of pages
per week.

**The action:** review flagged pages and decide whether to refresh, expand, protect, prune, or
monitor them.

**Cost of a wrong call:**
- False positive (flagged but not actually a problem): wasted reviewer time, low cost.
- False negative (a genuinely declining page never flagged): a real opportunity missed, page
  keeps losing visibility unreviewed.
- Because review capacity is limited, Precision@K matters more than overall accuracy — the
  reviewer only ever sees the top of the ranked list.

**One-paragraph frame:** For a content/SEO reviewer, deciding which pages to review first out of
a large inventory, we will build a ranked priority queue from observable search and content
signals, scoring pages by decline risk, measured by Precision@K against real review capacity. A
wrong call costs wasted reviewer time or a missed declining page. A plain rule isn't enough
because the signals are tangled — content_age_days, content_type, and client all show real,
partly independent effects, which a simple hand rule struggles to combine well. We will claim
only observed, directional, decision-support results.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Supporting check: does the starter baseline's own logic (staleness-based) line up with
# what the data actually shows about age and decline?
corr_age = df["content_age_days"].corr(df["is_declining"])
print(f"Correlation between content_age_days and is_declining: {corr_age:.3f}")
print("Negative correlation -> younger pages decline more, the opposite of a pure 'staleness' rule")

Correlation between content_age_days and is_declining: -0.164
Negative correlation -> younger pages decline more, the opposite of a pure 'staleness' rule


## 3. Quick look at the data (2-3 real numbers)

Three numbers that make this lane worth building on:

In [3]:
!pip install -q reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.4 MB/s eta 0:00:00


In [4]:
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank-Internship-Work/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/FlyRank-Internship-Work/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/FlyRank-Internship-Work/data/processed/model_predictions.csv
Wrote model results: /content/FlyRank-Internship-Work/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/FlyRank-Internship-Work/outputs/refresh_queue.csv
Wrote model report: /co

In [5]:
# Number 1: content_type effect on decline rate — real and repeats across age tiers
print("--- Decline rate by content_type ---")
print(df.groupby("content_type")["is_declining"].mean().round(3).sort_values(ascending=False))

# Number 2: client concentration — decline is not evenly spread across clients
client_decline = df.groupby("client_id")["is_declining"].mean()
print(f"\nDecline rate across clients — mean: {client_decline.mean():.3f}, std dev: {client_decline.std():.3f}")
print(f"Worst client: {client_decline.max():.3f}   Best client: {client_decline.min():.3f}")
print(f"Clients with >70% decline rate: {(client_decline > 0.7).sum()} / {len(client_decline)}")

# Number 3: evidence a learned rank can meaningfully beat a hand rule on this kind of task
import json
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"\nBaseline Precision@50: {base:.3f}   Random forest Precision@50: {rf:.3f}   (~{rf/base:.1f}x lift)")

--- Decline rate by content_type ---
content_type
comparison article    0.572
keyword article       0.561
feedly article        0.287
Name: is_declining, dtype: float64

Decline rate across clients — mean: 0.489, std dev: 0.227
Worst client: 0.937   Best client: 0.000
Clients with >70% decline rate: 4 / 32

Baseline Precision@50: 0.240   Random forest Precision@50: 0.740   (~3.1x lift)


## 4. Careful words: what I can and can't claim

**What I can say:** this work is observational and decision-support only. I can describe
patterns I observe (e.g. decline skews toward younger pages and certain content types, and is
unevenly concentrated across clients), and use those patterns to build a ranked, directional
priority list for human review. A higher score means "more worth a reviewer's time," not
"guaranteed to be a problem."

**What I cannot say:** I cannot claim this predicts or reverse-engineers Google's ranking
algorithm. I cannot claim that refreshing a flagged page will cause it to recover — that would
require a controlled experiment this data doesn't provide. The label I'm using
(`trend_direction == "down"`) is a proxy based on a current-state bucket, not a validated future
outcome — a stronger version I plan to build toward uses a real prior-window → future-window
label once I move to the warehouse data. I'll also rule out consolidation, seasonality, and
low-volume noise before calling anything a confident "decline," and I'll account for client-level
concentration rather than treating every page as an independent, interchangeable case.## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [6]:
print(df.groupby("content_type")["is_declining"].agg(["mean", "count"]))
print("\nSmallest group (comparison article) still has 697 rows -> not a small-sample artifact")

                        mean  count
content_type                       
comparison article  0.572453    697
feedly article      0.286737   2096
keyword article     0.560959  27207

Smallest group (comparison article) still has 697 rows -> not a small-sample artifact


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.